#### Imports y path:

In [1]:
import os
import sys
from pathlib import Path

# Ajusta al root del repo Bluegrey si hace falta
ROOT = Path.cwd()
if (ROOT / "tools").exists():
    repo_root = ROOT
else:
    # si estás dentro de research/, sube un nivel
    repo_root = ROOT.parent

sys.path.append(str(repo_root))
print("Repo root:", repo_root)

Repo root: /Users/marina/Documents/jordi/projects/Trading/bluegrey/0. Discover/Bluegrey


#### Crea el ingestor y descarga
- El script ya trae una clase lista: PolygonIngestor.
- Descubre pares FX líquidos (fetch_liquid_fx_tickers)
- Descarga velas de 1 minuto (list_aggs)
- Guarda en ArcticDB (write/update)

In [5]:
from src.infra.config import UNIVERSE_FX
UNIVERSE_FX = ['C:' + key for key in UNIVERSE_FX]
UNIVERSE_FX

['C:EURUSD', 'C:GBPUSD', 'C:AUDUSD', 'C:NZDUSD', 'C:USDJPY']

In [6]:
from tools.download_history_polygon import PolygonIngestor

ing = PolygonIngestor()

# Ver algunos pares que detecta
pairs = ing.fetch_liquid_fx_tickers()
print("Total pares:", len(pairs))
print("Ejemplo:", pairs[:10])

# Descargar solo unos pocos para prueba rápida
#for t in pairs[:3]:
for t in UNIVERSE_FX:
    ing.download_ticker(t, start_year=2024)

2026-04-07 15:42:28,774 [INFO] Querying Polygon for FX pairs (Liquid Filter Active)...
2026-04-07 15:42:29,747 [INFO] Filter Complete: Reduced universe from 1000+ to 167 high-quality pairs.
Total pares: 167
Ejemplo: ['C:AUDCAD', 'C:AUDCHF', 'C:AUDEUR', 'C:AUDGBP', 'C:AUDHKD', 'C:AUDJPY', 'C:AUDMXN', 'C:AUDNOK', 'C:AUDNZD', 'C:AUDSEK']
✅ Created C:EURUSD: 834893 bars.
✅ Created C:GBPUSD: 832904 bars.
✅ Updated C:AUDUSD: 828648 bars.
✅ Created C:NZDUSD: 837179 bars.
✅ Created C:USDJPY: 833085 bars.


#### Check que se guardó en ArcticDB

In [9]:
# Verifica que se guardó en ArcticDB

from src.infra.store import DataStore
import src.infra.config as config

store = DataStore(config.LIBS["fx_min"])  # normalmente "fx.min"
print("OK store:", config.LIBS["fx_min"])

2026-04-07 17:13:08,226 [INFO] 🗄️ Connected to ArcticDB at: lmdb:///Users/marina/Documents/jordi/projects/Trading/bluegrey/0. Discover/Bluegrey/data/arctic_db?map_size=10GB
OK store: fx.min


20260407 17:13:08.225508 7027822 W arcticdb | LMDB path at /Users/marina/Documents/jordi/projects/Trading/bluegrey/0. Discover/Bluegrey/data/arctic_db/ has already been opened in this process which is not supported by LMDB. You should only open a single Arctic instance over a given LMDB path. To continue safely, you should delete this Arctic instance and any others over the LMDB path in this process and then try again. Current process ID=[20604]


In [10]:
df = store.load("C:AUDCAD", start_date="2024-01-01")
print(df.head())
print(df.tail())
print("Rows:", len(df))

                         open      high       low     close  volume    vwap
timestamp                                                                  
2024-01-01 00:32:00  0.902200  0.902339  0.902200  0.902339       2  0.9023
2024-01-01 00:33:00  0.902100  0.902100  0.901461  0.901461       2  0.9018
2024-01-01 00:34:00  0.902000  0.902100  0.901961  0.901961       3  0.9020
2024-01-01 08:31:00  0.902100  0.902100  0.902100  0.902100       1  0.9021
2024-01-01 09:59:00  0.902445  0.902445  0.902000  0.902000       3  0.9022
                        open      high     low    close  volume    vwap
timestamp                                                              
2026-04-07 13:12:00  0.96531  0.965567  0.9649  0.96542     229  0.9654
2026-04-07 13:13:00  0.96548  0.965654  0.9649  0.96554     213  0.9654
2026-04-07 13:14:00  0.96556  0.965680  0.9648  0.96545     303  0.9654
2026-04-07 13:15:00  0.96550  0.965658  0.9649  0.96552     231  0.9655
2026-04-07 13:16:00  0.96548  0.9655